# Notebook 5: Dual-Channel Simultaneous XADC Acquisition Test

This notebook verifies the **Simultaneous Dual-Channel XADC architecture** on the PYNQ-Z2 (`v1.3.0-rc1`).

### 🔌 Hardware Wiring Verification:
* **AD3 W1 (Solid Yellow)** $\rightarrow$ **PYNQ-Z2 A0** (Pin 6 - Bottom Pin on Analog Header `J1`)
* **AD3 W2 (Yellow/White)** $\rightarrow$ **PYNQ-Z2 A1** (Pin 5 - 2nd Pin from Bottom on `J1`)
* **AD3 GND (Solid Black)** $\rightarrow$ **PYNQ-Z2 GND**

## 1. System Setup & Permission Check

In [ ]:
from pynq_oscilloscope import check_usb_permissions, OscilloscopeOverlay
from pydwf import DwfLibrary, DwfAnalogOutFunction, DwfAnalogOutNode
from pydwf.utilities import openDwfDevice
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pynq import allocate
import numpy as np
import time

# Ensure USB permissions for AD3 communication
check_usb_permissions()

# Clean any dangling AD3 USB handles
dwf = DwfLibrary()
try:
    dwf.deviceControl.closeAll()
except Exception:
    pass

## 2. Load the Dual-Channel Hardware Overlay
Instantiating `OscilloscopeOverlay()` automatically loads `v1.3.0-rc1` containing the dual-ADC package routing (`E17/D18` and `E18/E19`).

In [ ]:
ol = OscilloscopeOverlay()

# Initialize XADC in Continuous Sequencer Mode for Vaux1 (A0) and Vaux9 (A1)
xadc = ol.xadc_wiz_0.mmio
xadc.write(0x304, 0x2000)  # DRP 0x41 = 0x2000 (Continuous Sequence Mode)
xadc.write(0x320, 0x0000)  # DRP 0x48 = Disable internal temp/voltage channels
xadc.write(0x324, 0x0202)  # DRP 0x49 = Enable Vaux1 (bit 1) and Vaux9 (bit 9)
time.sleep(0.05)

print("✅ Dual-Channel Oscilloscope Overlay loaded and XADC Sequencer initialized!")
print(f"   Config Reg 1 (0x304): 0x{xadc.read(0x304):04X} (Continuous Sequencer)")
print(f"   SEQ_CHSEL1   (0x324): 0x{xadc.read(0x324):04X} (Vaux1 & Vaux9 Active)")

## 3. Fast Static Pin Voltage Verification
Verifies that both physical analog pins (`A0` and `A1`) respond to distinct DC voltage levels before starting AC signals.

In [ ]:
with openDwfDevice(dwf) as device:
    ao = device.analogOut
    
    # Set W1 = 1.00V DC (A0), W2 = 2.50V DC (A1)
    ao.nodeEnableSet(0, DwfAnalogOutNode.Carrier, True)
    ao.nodeFunctionSet(0, DwfAnalogOutNode.Carrier, DwfAnalogOutFunction.DC)
    ao.nodeOffsetSet(0, DwfAnalogOutNode.Carrier, 1.00)
    
    ao.nodeEnableSet(1, DwfAnalogOutNode.Carrier, True)
    ao.nodeFunctionSet(1, DwfAnalogOutNode.Carrier, DwfAnalogOutFunction.DC)
    ao.nodeOffsetSet(1, DwfAnalogOutNode.Carrier, 2.50)
    
    ao.configure(-1, True)
    time.sleep(0.3)
    
    raw_a0 = (xadc.read(0x244) >> 4) & 0xFFF
    raw_a1 = (xadc.read(0x264) >> 4) & 0xFFF
    v_a0 = raw_a0 * (3.3 / 4095.0)
    v_a1 = raw_a1 * (3.3 / 4095.0)
    
    print("=== Static Voltage Check ===")
    print(f"  • A0 (Target W1: 1.00V) -> Measured: {v_a0:.2f} V (Code: 0x{raw_a0:03X})")
    print(f"  • A1 (Target W2: 2.50V) -> Measured: {v_a1:.2f} V (Code: 0x{raw_a1:03X})")
    
    ao.configure(-1, False)

assert abs(v_a0 - 1.00) < 0.2, "Error: A0 did not detect 1.00V from W1!"
assert abs(v_a1 - 2.50) < 0.2, "Error: A1 did not detect 2.50V from W2!"
print("🎉 Physical Pin Mapping & DC Tracking 100% Verified!")

## 4. Generate Dual-Channel AC Test Signals
We start two distinct test waveforms:
* **Channel 1 (W1 $\rightarrow$ A0):** $1\,\text{kHz}$ Sine wave ($1.0\,\text{V}$ Amplitude, $1.65\,\text{V}$ Offset)
* **Channel 2 (W2 $\rightarrow$ A1):** $5\,\text{kHz}$ Square wave ($1.0\,\text{V}$ Amplitude, $1.65\,\text{V}$ Offset)

In [ ]:
# Start Dual-Channel Signal Generation
ol.wavegen.start(
    shape="Sine", frequency=1000.0, amplitude=1.0, offset=1.65,        # Channel 1 (W1 -> A0)
    ch2_shape="Square", ch2_frequency=5000.0, ch2_amplitude=1.0, ch2_offset=1.65, # Channel 2 (W2 -> A1)
    enable_ch2=True
)
time.sleep(1.0)
print("✅ AD3 active: W1 (1 kHz Sine) -> A0, W2 (5 kHz Square) -> A1")

## 5. Configure Hardware Trigger
Configure the FPGA trigger unit to trigger on the rising edge of Channel 1 (A0) at $1.65\,\text{V}$.

In [ ]:
ol.trigger.configure(mode="Auto", edge="Rising", threshold_volts=1.65, timeout_ms=50.0)
print(f"Hardware Trigger Threshold: {ol.trigger.get_threshold():.2f} V")

## 6. Simultaneous Dual-Channel DMA Capture
Capture both channels synchronously via the hardened dual-DMA sequence.

In [ ]:
def capture_stereo_sync(overlay, packet_size=2048):
    dma_time = overlay.axi_dma_0
    dma_fft = overlay.axi_dma_1
    trig = overlay.trigger
    
    # 1. Reset DMAs to clean state
    dma_time.mmio.write(0x30, 0x04)
    dma_fft.mmio.write(0x30, 0x04)
    time.sleep(0.01)
    dma_time.recvchannel.start()
    dma_fft.recvchannel.start()
    
    # 2. Allocate buffers
    buf_time = allocate(shape=(packet_size,), dtype="u2")
    buf_fft = allocate(shape=(packet_size,), dtype="u2")
    
    try:
        # 3. Queue both DMAs first (Prevents Broadcaster Deadlock)
        dma_time.recvchannel.transfer(buf_time)
        dma_fft.recvchannel.transfer(buf_fft)
        
        # 4. Arm trigger in Single-Shot Auto Mode (0x0B)
        trig.mmio.write(0x00, 0x0B)
        
        # 5. Wait for hardware completion
        t_start = time.time()
        while not (dma_time.recvchannel.idle and dma_fft.recvchannel.idle):
            time.sleep(0.001)
            if time.time() - t_start > 0.5:
                trig.force_trigger()
                time.sleep(0.01)
                break
                
        # 6. De-interleave: Even = A0 (Ch1), Odd = A1 (Ch2)
        raw = np.array(buf_time)
        ch1_v = (raw[0::2] >> 4) * (3.3 / 4095.0)
        ch2_v = (raw[1::2] >> 4) * (3.3 / 4095.0)
        
        return ch1_v, ch2_v
        
    finally:
        buf_time.close()
        buf_fft.close()

# Capture dual channels
v_ch1, v_ch2 = capture_stereo_sync(ol)

print(f"✅ Captured {len(v_ch1)} simultaneous sample pairs!")
print(f"  • CH1 (A0 - Sine)   : Min = {v_ch1.min():.2f} V, Max = {v_ch1.max():.2f} V, Vpp = {v_ch1.max()-v_ch1.min():.2f} V")
print(f"  • CH2 (A1 - Square) : Min = {v_ch2.min():.2f} V, Max = {v_ch2.max():.2f} V, Vpp = {v_ch2.max()-v_ch2.min():.2f} V")

## 7. Dual-Channel Waveform Plot (Matplotlib)
Plot both channels side-by-side to verify time alignment and distinct waveforms.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, dpi=100)
time_us = np.arange(len(v_ch1))

# Channel 1 (A0)
ax1.plot(time_us[:500], v_ch1[:500], color="#00FFCC", linewidth=1.8, label="CH1: A0 (1 kHz Sine)")
ax1.axhline(1.65, color="#FFA500", linestyle="--", alpha=0.7, label="Trigger (1.65V)")
ax1.set_ylabel("Voltage (V)", fontsize=10)
ax1.set_ylim(0.0, 3.3)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(loc="upper right")
ax1.set_title("Dual-Channel Simultaneous Acquisition (1 MSPS)", fontsize=12, fontweight="bold")

# Channel 2 (A1)
ax2.plot(time_us[:500], v_ch2[:500], color="#FF007F", linewidth=1.8, label="CH2: A1 (5 kHz Square)")
ax2.set_xlabel("Time (Microseconds @ 1 MSPS)", fontsize=10)
ax2.set_ylabel("Voltage (V)", fontsize=10)
ax2.set_ylim(0.0, 3.3)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 8. Interactive Plotly Dual-Trace Scope View

In [ ]:
fig_plotly = go.Figure()
fig_plotly.add_trace(go.Scatter(y=v_ch1[:500], mode='lines', name='CH1: A0 (Sine)', line=dict(color='#00FFCC', width=2)))
fig_plotly.add_trace(go.Scatter(y=v_ch2[:500], mode='lines', name='CH2: A1 (Square)', line=dict(color='#FF007F', width=2)))
fig_plotly.add_hline(y=1.65, line_dash="dash", line_color="#FFA500", annotation_text="Trigger Threshold (1.65V)")

fig_plotly.update_layout(
    title="<b>Interactive Dual-Channel Oscilloscope Stream</b>",
    xaxis_title="Time (Microseconds @ 1 MSPS)",
    yaxis_title="Voltage (V)",
    yaxis_range=[0.0, 3.3],
    template="plotly_dark",
    height=450
)
fig_plotly.show()

## 9. Clean Hardware Shutdown

In [ ]:
ol.close()
print("🔒 AD3 Wavegen stopped and overlay memory cleanly released.")